In [1]:
import pandas as pd

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train['text'], train['generated'], test_size=0.1, random_state=42)
valid = pd.concat([X_test, y_test], axis=1)

In [4]:
print(f"Среднее значение таргета = {train['generated'].mean()}")
print(f"Размер тренировочной выборки = {train['generated'].shape[0]}")

Среднее значение таргета = 0.3722613318008764
Размер тренировочной выборки = 389788


In [5]:
train['text'].iloc[1102]

'Cars most people use them for transportation. It\'s our lazy way of getting to places.\n\nAutomobiles benefit us plenty when it comes to getting from point A to point B, but they also have a big negative impact. Cars are responsible for a huge amount of pollution like greenhouse gas emissions and smog. If we were to limit car usage, we could decrease the amount of stress and pollution emitted into the air, as well as give our community a chance to improve.\n\nIn Vauban, Germany, residents have given UC their cars and have no problem doing so. Cars are generally not allowed, forbidden some would say, in this district. Vauban\'s streets are pretty much "carefree". Of course car ownership is still allowed, with the accession that you have to be able to find a place to car since there are only two places; large garages at the end of the development, where a car owner can buy a scale for $40,000, along with a home. "When I had a car I was always tense. I\'m much hacker this way," Hadron Wa

In [6]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import pandas as pd
import numpy as np

from collections import Counter
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# import matplotlib.pyplot as plt
# import seaborn as sns

In [ ]:
# import string

# # Скачивание необходимых ресурсов NLTK (если нужно)
# # nltk.download('stopwords')
# # nltk.download('wordnet')
# # nltk.download('punkt')

# def preprocess_text(text, language='english'):
#     """
#     Основная функция предобработки текста
#     """
#     if not isinstance(text, str):
#         return ""
    
#     # 1. Приведение к нижнему регистру
#     text = text.lower()
    
#     text = re.sub(r'{\n}', ' ', text)
    
#     # 6. Удаление пунктуации
#     text = text.translate(str.maketrans('', '', string.punctuation))
    
#     # 7. Удаление лишних пробелов
#     text = re.sub(r'\s+', ' ', text).strip()
    
#     return text

def advanced_preprocessing(text, remove_stopwords=True, lemmatize=True):
    
    # Токенизация
    tokens = nltk.word_tokenize(text)
    
    # # Удаление стоп-слов
    # if remove_stopwords:
    #     stop_words = set(stopwords.words('english'))
    #     tokens = [word for word in tokens if word not in stop_words]
    
    # # Лемматизация
    # if lemmatize:
    #     lemmatizer = WordNetLemmatizer()
    #     tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

In [7]:
from tqdm import tqdm
tqdm.pandas()

# train['cleaned_text'] = train['text'].progress_apply(preprocess_text)
train['processed_text'] = train['text'].progress_apply(lambda text: ' '.join(nltk.word_tokenize(text)))

100%|██████████| 389788/389788 [09:57<00:00, 652.76it/s]


In [11]:
train.to_csv("train_preprocessing.csv")

In [12]:
valid['processed_text'] = valid['text'].progress_apply(lambda text: ' '.join(nltk.word_tokenize(text)))
test['processed_text'] = test['text'].progress_apply(lambda text: ' '.join(nltk.word_tokenize(text)))

valid.to_csv("valid_preprocessing.csv")
test.to_csv("test_preprocessing.csv")

100%|██████████| 97447/97447 [02:28<00:00, 656.41it/s]


In [9]:
train[['text', 'processed_text']].head()

,text,processed_text
0,"I think that FACS is very useful technology, t...","I think that FACS is very useful technology , ..."
1,Should students create their own summer projec...,Should students create their own summer projec...
2,"As an average 8thgrade student, I have develop...","As an average 8thgrade student , I have develo..."
3,Holy Avocados! A new computer software has jus...,Holy Avocados ! A new computer software has ju...
4,Title: A Cowboy Who Rode the Waves\n\nOnce upo...,Title : A Cowboy Who Rode the Waves Once upon ...


In [24]:
# dvc_loader.py
import subprocess
import pandas as pd
from pathlib import Path

def load_from_gdrive(dvc_path, encoding='utf-8'):
    """Автоматически скачивает файл из Google Drive если его нет"""
    local_path = Path(dvc_path)
    
    # Если файла нет - скачиваем через dvc pull
    if not local_path.exists():
        print(f"Скачиваю {dvc_path} из Google Drive...")
        result = subprocess.run(['dvc', 'pull', dvc_path], 
                              capture_output=True, text=True)
        if result.returncode != 0:
            raise Exception(f"Ошибка: {result.stderr}")
    
    # Читаем файл
    return pd.read_csv(local_path, encoding=encoding)

df = load_from_gdrive('data/test.csv')

Скачиваю data/test.csv из Google Drive...


Exception: Ошибка: WARNING: failed to collect 'workspace', skipping
ERROR: failed to pull data from the cloud - 'data/test.csv' does not exist as an output or a stage name in 'dvc.yaml': 'dvc.yaml' does not exist


In [22]:
import dvc.api

# Путь к файлу в DVC репозитории
data_path = 'data/test.csv'

with dvc.api.open(data_path, repo='.', encoding='utf-8') as fd:
    df = pd.read_csv(fd)

PathMissingError: The path 'data/test.csv' does not exist in the target repository 'data/test.csv' neither as a DVC output nor as a Git-tracked file.

In [21]:
df

,text,generated
0,"Real or Fake Feelings\n\n""Imagine being able t...",0.0
1,Seeking multiple opinions can help you make be...,0.0
2,"ADDRESS_NAME\n\nFebruary 9, 2011\n\nDear TEACH...",0.0
3,"Dear, TEACHER_NAME,\n\nTEACHER_NAME I Believe ...",0.0
4,Do you believe that there is a computer that c...,0.0
...,...,...
97442,"Are Senator,\n\nI am writing to you today to e...",1.0
97443,Car usage has been a popular mode of Transpor...,1.0
97444,The author suggests that studying Venus is a w...,0.0
97445,With more and more schools offering home schoo...,0.0


In [ ]:
# config_loader.py
import yaml
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

@dataclass
class PathsConfig:
    data_path: str
    model_save_path: str
    pretrained_embeddings: str
    vocab_save_path: str
    
@dataclass
class PreprocessingConfig:
    max_length: int
    min_word_freq: int
    vocab_size: int
    stats_features: List[str]
    
@dataclass
class TrainingConfig:
    batch_size: int
    num_workers: int
    test_size: float
    val_size: float
    random_seed: int
    max_epochs: int
    learning_rate: float
    patience: int
    
@dataclass
class ModelConfig:
    embedding_dim: int
    hidden_dim: int
    num_layers: int
    dropout: float
    bidirectional: bool
    use_pretrained_embeddings: bool
    
@dataclass
class FeaturesConfig:
    normalize_text_length: int
    normalize_counts: int
    tokenizer: str

@dataclass
class Config:
    """Основной класс конфигурации"""
    paths: PathsConfig
    preprocessing: PreprocessingConfig
    training: TrainingConfig
    model: ModelConfig
    features: FeaturesConfig
    
    @classmethod
    def from_yaml(cls, config_path: str):
        """Загрузка конфигурации из YAML файла"""
        with open(config_path, 'r', encoding='utf-8') as f:
            config_dict = yaml.safe_load(f)
        
        return cls(
            paths=PathsConfig(**config_dict['paths']),
            preprocessing=PreprocessingConfig(**config_dict['preprocessing']),
            training=TrainingConfig(**config_dict['training']),
            model=ModelConfig(**config_dict['model']),
            features=FeaturesConfig(**config_dict['features'])
        )
    
    def to_dict(self) -> Dict[str, Any]:
        """Конвертация в словарь"""
        return {
            'paths': self.paths.__dict__,
            'preprocessing': self.preprocessing.__dict__,
            'training': self.training.__dict__,
            'model': self.model.__dict__,
            'features': self.features.__dict__
        }

In [10]:
# dataset.py
import torch
from torch.utils.data import Dataset
from collections import Counter
from typing import List, Optional, Dict
from config_loader import Config

class AIDetectionDataset(Dataset):    
    def __init__(
        self, 
        texts: List[str], 
        labels: List[int],
        config: Config
    ):
        self.texts = texts
        self.labels = labels
        self.config = config
        
        self.vocab = self._build_vocab()
    
    def _build_vocab(self) -> Dict:
        all_tokens = []
        for item in self.processed_data:
            all_tokens.extend(item['tokens'])
        
        counter = Counter(all_tokens)
        
        vocab = {
            '<PAD>': 0,
            '<UNK>': 1,
            '<SOS>': 2,
            '<EOS>': 3
        }
        
        sorted_tokens = sorted(counter.items(), key=lambda x: x[1], reverse=True)
        for token, freq in sorted_tokens:
            if len(vocab) > self.config['training'].max_size_tokens:
                break
            vocab[token] = len(vocab)
        
        return vocab
    
    def _text_to_indices(self, tokens: List[str]) -> torch.Tensor:
        indices = []
        for token in tokens[:self.max_length]:
            indices.append(self.vocab.get(token, self.vocab['<UNK>']))
        
        indices = [self.vocab['<SOS>']] + indices + [self.vocab['<EOS>']]
        
        return torch.tensor(indices, dtype=torch.long)
    
    def __len__(self) -> int:
        return len(self.texts)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.processed_data[idx]
        
        token_indices = self._text_to_indices(item['tokens'])        
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return {
            'token_indices': token_indices,
            'label': label_tensor
        }
    
    def get_vocab_size(self) -> int:
        return len(self.vocab)


ModuleNotFoundError: No module named 'config_loader'